In [1]:
import math


# ----------------------------
# Lab 4 (Interpolation)
# Variant 21: y = ln(x), find y at x = 3.4
# Given nodes: 3.0, 3.5, 4.0, 4.5
# No external libraries (only math)
# ----------------------------

X_STAR = 3.4
X_NODES = [3.0, 3.5, 4.0, 4.5]


In [2]:

def f(x: float) -> float:
    return math.log(x)


def lagrange_value(x_nodes, y_nodes, x: float) -> float:
    n = len(x_nodes)
    s = 0.0
    for i in range(n):
        li = 1.0
        xi = x_nodes[i]
        for j in range(n):
            if j == i:
                continue
            xj = x_nodes[j]
            li *= (x - xj) / (xi - xj)
        s += y_nodes[i] * li
    return s


def lagrange_coeffs_cubic(x_nodes, y_nodes):
    """
    Build coefficients of cubic polynomial P(x)=a3*x^3+a2*x^2+a1*x+a0
    that interpolates points (x_nodes[i], y_nodes[i]), i=0..3.

    Done via explicit Lagrange basis expansion into coefficients.
    No numpy.
    """
    if len(x_nodes) != 4 or len(y_nodes) != 4:
        raise ValueError("This helper builds only cubic for 4 nodes.")

    # Polynomial represented as list [c0, c1, c2, c3] for c0 + c1*x + c2*x^2 + c3*x^3
    def poly_mul(p, q):
        r = [0.0] * (len(p) + len(q) - 1)
        for i in range(len(p)):
            for j in range(len(q)):
                r[i + j] += p[i] * q[j]
        return r

    def poly_scale(p, k):
        return [c * k for c in p]

    def poly_add(p, q):
        m = max(len(p), len(q))
        r = [0.0] * m
        for i in range(m):
            if i < len(p):
                r[i] += p[i]
            if i < len(q):
                r[i] += q[i]
        return r

    coeffs = [0.0, 0.0, 0.0, 0.0]

    # L_i(x) = Π_{j!=i} (x - xj) / (xi - xj)
    # numerator poly starts as [1]
    for i in range(4):
        xi = x_nodes[i]
        denom = 1.0
        num_poly = [1.0]  # 1

        for j in range(4):
            if j == i:
                continue
            xj = x_nodes[j]
            denom *= (xi - xj)
            # multiply by (x - xj) -> [-xj, 1]
            num_poly = poly_mul(num_poly, [-xj, 1.0])

        Li_poly = poly_scale(num_poly, 1.0 / denom)
        term = poly_scale(Li_poly, y_nodes[i])
        coeffs = poly_add(coeffs, term)

    # coeffs = [a0, a1, a2, a3]
    a0, a1, a2, a3 = coeffs[0], coeffs[1], coeffs[2], coeffs[3]
    return a3, a2, a1, a0


def omega_value(x_nodes, x: float) -> float:
    w = 1.0
    for xi in x_nodes:
        w *= (x - xi)
    return w


def f4_abs_max_ln(a: float, b: float) -> float:
    # f(x)=ln x, f^(4)(x)= -6/x^4, so |f^(4)(x)| = 6/x^4
    # maximum on [a,b] at x=a (since decreasing with x)
    return 6.0 / (a ** 4)



In [3]:
def main():
    # 1) Build table
    y_nodes = [f(x) for x in X_NODES]

    print("Lab 4. Variant 21")
    print("f(x) = ln(x)")
    print("Nodes:", X_NODES)
    print(f"x* = {X_STAR}")
    print()

    print("Table (x, ln(x)):")
    for x, y in zip(X_NODES, y_nodes):
        print(f"  x = {x:.1f}, ln(x) = {y:.10f}")
    print()

    # 2) Lagrange interpolation value
    y_lagr = lagrange_value(X_NODES, y_nodes, X_STAR)

    # 3) Exact value and absolute error
    y_exact = f(X_STAR)
    abs_err = abs(y_exact - y_lagr)

    print(f"Exact y(x*) = ln({X_STAR}) = {y_exact:.12f}")
    print(f"Lagrange L3(x*)        = {y_lagr:.12f}")
    print(f"Absolute error         = {abs_err:.12e}")
    print()

    # 4) Interpolating polynomial coefficients
    a3, a2, a1, a0 = lagrange_coeffs_cubic(X_NODES, y_nodes)
    print("Interpolating cubic polynomial:")
    print(f"  P(x) = {a3:.12f}*x^3 + {a2:.12f}*x^2 + {a1:.12f}*x + {a0:.12f}")
    print()

    # 5) Theoretical remainder bound
    # |R3(x)| <= (M4/4!) * |omega(x)|
    M4 = f4_abs_max_ln(min(X_NODES), max(X_NODES))
    omega = omega_value(X_NODES, X_STAR)
    bound = (M4 / 24.0) * abs(omega)

    print("Remainder bound for Lagrange interpolation:")
    print(f"  M4 = max|f^(4)(x)| on [{min(X_NODES)}, {max(X_NODES)}] = {M4:.12f}")
    print(f"  omega(x*) = Π(x* - xi) = {omega:.12f}")
    print(f"  |R3(x*)| <= (M4/4!)*|omega| = {bound:.12e}")
    print()

    # 6) Optional: compare bound vs actual error
    print("Check:")
    print(f"  actual error = {abs_err:.12e}")
    print(f"  bound        = {bound:.12e}")


if __name__ == "__main__":
    main()

Lab 4. Variant 21
f(x) = ln(x)
Nodes: [3.0, 3.5, 4.0, 4.5]
x* = 3.4

Table (x, ln(x)):
  x = 3.0, ln(x) = 1.0986122887
  x = 3.5, ln(x) = 1.2527629685
  x = 4.0, ln(x) = 1.3862943611
  x = 4.5, ln(x) = 1.5040773968

Exact y(x*) = ln(3.4) = 1.223775431622
Lagrange L3(x*)        = 1.223738245274
Absolute error         = 3.718634847316e-05

Interpolating cubic polynomial:
  P(x) = 0.006494573646*x^3 + -0.109431597690*x^2 + 0.813404031374*x + -0.532068914690

Remainder bound for Lagrange interpolation:
  M4 = max|f^(4)(x)| on [3.0, 4.5] = 0.074074074074
  omega(x*) = Π(x* - xi) = -0.026400000000
  |R3(x*)| <= (M4/4!)*|omega| = 8.148148148148e-05

Check:
  actual error = 3.718634847316e-05
  bound        = 8.148148148148e-05


In [6]:
import math

# -----------------------------
# Lab 4, Task 2 (like Shnitko)
# f(x) = cos(3x), [a,b] = [0,1]
# Table for k=0..m with m=4 (5 nodes), h=(b-a)/m
#
# Columns:
# x_{k,m} | f'(x) left | f'(x) right | f'(x) center | f''(x) (numeric) | f'(x) exact | f''(x) exact
#
# Plus absolute errors (optional table) like in the reference.
# -----------------------------

a = 0.0
b = 1.0
m = 4
h = (b - a) / m

def f(x: float) -> float:
    return math.cos(3.0 * x)

def fp_exact(x: float) -> float:
    return -3.0 * math.sin(3.0 * x)

def fpp_exact(x: float) -> float:
    return -9.0 * math.cos(3.0 * x)

# Nodes
x = [a + k * h for k in range(m + 1)]
y = [f(xk) for xk in x]

def fmt_num(v):
    return "-" if v is None else f"{v:.4f}"

def fmt_err(v):
    return "-" if v is None else f"{v:.4f}"

# Compute derivatives at nodes
left = [None] * (m + 1)
right = [None] * (m + 1)
center = [None] * (m + 1)
second = [None] * (m + 1)

for k in range(m + 1):
    if k >= 1:
        left[k] = (y[k] - y[k - 1]) / h
    if k <= m - 1:
        right[k] = (y[k + 1] - y[k]) / h
    if 1 <= k <= m - 1:
        center[k] = (y[k + 1] - y[k - 1]) / (2.0 * h)
        second[k] = (y[k + 1] - 2.0 * y[k] + y[k - 1]) / (h * h)

# Exact derivatives at nodes
exact_fp = [fp_exact(xk) for xk in x]
exact_fpp = [fpp_exact(xk) for xk in x]

# Print main table
print("2. Дифференцирование")
print(f"f(x)=cos(3x), a={a}, b={b}, m={m}, h={h}")
print()

header = (
    "x_k,m".ljust(10) + " | " +
    "f'(x) слева".ljust(12) + " | " +
    "f'(x) справа".ljust(13) + " | " +
    "f'(x) центр".ljust(12) + " | " +
    "f''(x)".ljust(10) + " | " +
    "f'(x) точн".ljust(11) + " | " +
    "f''(x) точн".ljust(12)
)
print(header)
print("-" * len(header))

for k in range(m + 1):
    xkm = f"{x[k]:.3f},{k}".ljust(10)  # like "0,000, 0" but with dot; keep style close
    row = (
        xkm + " | " +
        fmt_num(left[k]).ljust(12) + " | " +
        fmt_num(right[k]).ljust(13) + " | " +
        fmt_num(center[k]).ljust(12) + " | " +
        fmt_num(second[k]).ljust(10) + " | " +
        f"{exact_fp[k]:.4f}".ljust(11) + " | " +
        f"{exact_fpp[k]:.4f}".ljust(12)
    )
    print(row)

print()
print("Оценим абсолютные погрешности:")
header2 = (
    "x_k,m".ljust(10) + " | " +
    "f'(x) слева".ljust(12) + " | " +
    "f'(x) справа".ljust(13) + " | " +
    "f'(x) центр".ljust(12) + " | " +
    "f''(x) числ".ljust(12)
)
print(header2)
print("-" * len(header2))

for k in range(m + 1):
    err_left = None if left[k] is None else abs(left[k] - exact_fp[k])
    err_right = None if right[k] is None else abs(right[k] - exact_fp[k])
    err_center = None if center[k] is None else abs(center[k] - exact_fp[k])
    err_second = None if second[k] is None else abs(second[k] - exact_fpp[k])

    xkm = f"{x[k]:.3f},{k}".ljust(10)
    row2 = (
        xkm + " | " +
        fmt_err(err_left).ljust(12) + " | " +
        fmt_err(err_right).ljust(13) + " | " +
        fmt_err(err_center).ljust(12) + " | " +
        fmt_err(err_second).ljust(12)
    )
    print(row2)


2. Дифференцирование
f(x)=cos(3x), a=0.0, b=1.0, m=4, h=0.25

x_k,m      | f'(x) слева  | f'(x) справа  | f'(x) центр  | f''(x)     | f'(x) точн  | f''(x) точн 
--------------------------------------------------------------------------------------------------
0.000,0    | -            | -1.0732       | -            | -          | -0.0000     | -9.0000     
0.250,1    | -1.0732      | -2.6438       | -1.8585      | -6.2822    | -2.0449     | -6.5852     
0.500,2    | -2.6438      | -2.7956       | -2.7197      | -0.6073    | -2.9925     | -0.6366     
0.750,3    | -2.7956      | -1.4473       | -2.1215      | 5.3935     | -2.3342     | 5.6536      
1.000,4    | -1.4473      | -             | -            | -          | -0.4234     | 8.9099      

Оценим абсолютные погрешности:
x_k,m      | f'(x) слева  | f'(x) справа  | f'(x) центр  | f''(x) числ 
-----------------------------------------------------------------------
0.000,0    | -            | 1.0732        | -            | -         